In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json

import ollama

from app.data_loader import DataLoader
from app.schemas import Question, Answer, GradingResult
from app.grading import grade_answer
from app.concepts_extraction import extract_key_concepts
from app.prompts import get_template
from app.schemas import ModelEvaluation
from app.utils import write_file_with_uuid

In [3]:
test_cases = DataLoader("../data/generated_questions").test_cases

In [4]:
models = {
    "qwen2-7b": "qwen2.5:7b",
    "qwen2-3b": "qwen2.5:3b",
    "qwen3-8b": "qwen3:8b",
    "tlite-8b": "t-tech/T-lite-it-2.1:Q4_K_M",
    "saiga-8b": "bambucha/saiga-llama3:8b",
    "vikhr-8b": "rscr/vikhr_llama3.1_8b:Q4_K_M",
    "llama3-8b": "llama3:8b-instruct-q4_K_M",
    "mistral-7b": "mistral:7b",
    "phi-8b": "phi3.5:3.8b-mini-instruct-q4_0",
    "yi-6b": "yi:6b",
}

In [5]:
cases_to_test = [
    "08e8b29c583d4d3ca59749bcfb29fc7c",
    "0c5d8358c3c246958e6a1ccd80278b02",
    "10fd8fbf6ce2453a9dc909c54398123c",
    "111c5dd3e5204d5fa13c2afe9971f5f5",
    "122649a554f5404baf5c723c83c7db10",
]

In [6]:
cases = {k: test_cases[k] for k in test_cases if k in cases_to_test}

In [7]:
key_concepts = []

system_concept_template = get_template("concepts_extraction/system")
user_concept_template = get_template("concepts_extraction/user")

for case_uuid in cases:
    case = cases[case_uuid]
    
    concepts, _ = extract_key_concepts(
        "qwen2.5:3b",
        0.9,
        system_concept_template.render(),
        user_concept_template.render({"question": case.question}),
    )
    key_concepts.append(concepts)

In [8]:
key_concepts

[KeyConcepts(concepts=[Concept(text='Эйдос — это идеальная форма вещи, её сущность, существующая вне материального мира.', importance=9), Concept(text='Эйдосы являются первопричинами всего сущего и источником истинного знания.', importance=9), Concept(text='Человек познаёт мир через созерцание эйдосов, переходя от чувственного восприятия к интеллектуальному пониманию.', importance=8)]),
 KeyConcepts(concepts=[Concept(text='Личная гигиена — комплекс мероприятий, способствующих сохранению и укреплении здоровья.', importance=9), Concept(text='включает регулярную смену одежды', importance=7), Concept(text='мытьё рук перед едой и после посещения туалета, чистку зубов дважды в день, уход за кожей и волосами.', importance=8), Concept(text='Соблюдение правил личной гигиены предотвращает распространение инфекционных болезней', importance=9), Concept(text='снижает риск развития кожных инфекций, стоматологических проблем и кишечных расстройств.', importance=8)]),
 KeyConcepts(concepts=[Concept(te

In [9]:
system_grading_template = get_template("grading/system")

In [10]:
from tqdm.notebook import tqdm

In [12]:
temperatures = [0.5, 0.7, 0.9, 1, 1.2, 1.5]

temperature_pbar = tqdm(
    temperatures, total=len(temperatures), desc=f"Temperature", position=0, leave=True
)

for temperature in temperature_pbar:
    temperature_pbar.set_postfix({"t": temperature})

    model_pbar = tqdm(
        models,
        total=len(models),
        desc=f"Models",
        position=1,
        leave=True,
    )
    for model_short in model_pbar:
        model_pbar.set_postfix({"model": model_short})

        model = models[model_short]

        # print(f"{model_short} at t={temperature}")

        case_pbar = tqdm(
            enumerate(cases), total=len(cases), desc=f"Cases", position=2, leave=False
        )

        for case_idx, test_case_uuid in case_pbar:
            test_case = cases[test_case_uuid]

            concepts = key_concepts[case_idx]

            answer_pbar = tqdm(
                enumerate(test_case.answers),
                total=len(test_case.answers),
                desc=f"Answers for case {case_idx+1}",
                position=3,
                leave=False,
            )

            for answer_idx, answer in answer_pbar:
                try:
                    system_prompt = system_grading_template.render(
                        {
                            "question": test_case.question,
                            "key_concepts": concepts.concepts,
                        }
                    )
                    # print(system_prompt)
                    grading_result, response = grade_answer(
                        model,
                        temperature,
                        system_prompt,
                        answer.text,
                        len(concepts.concepts),
                    )
                except Exception as e:
                    print(e)
                    continue

                evaluation = ModelEvaluation(
                    case_uuid=test_case_uuid,
                    model=model_short,
                    temperature=temperature,
                    question=test_case.question,
                    key_concepts=concepts,
                    answer=answer,
                    grading_results=grading_result,
                    eval_duration=response.eval_duration,
                    eval_count=response.eval_count,
                )

                write_file_with_uuid(
                    f"../output/{model_short}/{temperature}",
                    evaluation.model_dump_json(indent=2),
                )

Temperature:   0%|          | 0/6 [00:00<?, ?it/s]

Models:   0%|          | 0/10 [00:00<?, ?it/s]

Cases:   0%|          | 0/5 [00:00<?, ?it/s]

Answers for case 1:   0%|          | 0/5 [00:00<?, ?it/s]

Answers for case 2:   0%|          | 0/5 [00:00<?, ?it/s]

...

Answers for case 2:   0%|          | 0/5 [00:00<?, ?it/s]

Unterminated string starting at: line 7 column 16 (char 402)


Answers for case 3:   0%|          | 0/5 [00:00<?, ?it/s]

Answers for case 4:   0%|          | 0/5 [00:00<?, ?it/s]

Answers for case 5:   0%|          | 0/5 [00:00<?, ?it/s]